In [ ]:
!pip install llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125

In [ ]:
# Create the target directory
!mkdir -p /content/models

# Download the model weights directly using huggingface_hub
!pip install -q huggingface_hub
!python3 -c "from huggingface_hub import hf_hub_download; hf_hub_download(repo_id='Qwen/Qwen2.5-3B-Instruct-GGUF', filename='qwen2.5-3b-instruct-q4_k_m.gguf', local_dir='/content/models')"

# Match your exact local file path variable
!mv /content/models/qwen2.5-3b-instruct-q4_k_m.gguf /content/models/qwen3.5-4b.Q4_K_M.gguf

In [ ]:
from llama_cpp import Llama

# ------------------------------------------------------------
# 3. LOAD LLAMA.CPP QWEN GGUF MODEL
# ------------------------------------------------------------

# Path to the Qwen GGUF model file
MODEL_PATH = "/content/models/qwen3.5-4b.Q4_K_M.gguf"

# Initialize Llama model with a larger context window
# n_ctx: Maximum context window size in tokens.
# verbose: Set to True for more detailed output from llama-cpp.
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048, # Increased context window to handle longer documents
    n_gpu_layers=-1, # Offload all layers to GPU if available
    verbose=False # Set to True for debugging llama.cpp internal logs
)

print("Llama model loaded successfully.")

# ------------------------------------------------------------
# 5. PREDICT A CLASS USING GENERATIVE MODEL
# ------------------------------------------------------------

def predict_digit(prompt_text, allowed_digits):
    """
    Generates a single digit prediction from a generative model based on the prompt.
    The model is instructed to output only the digit.
    """
    # Create a completion request to the generative model
    response = llm.create_completion(
        prompt=prompt_text,
        max_tokens=1,  # We expect only a single digit as output
        temperature=0.0, # Use a low temperature for deterministic output
        stop=["\n", " "] # Stop generation at newline or space to ensure single digit
    )

    # Extract the generated text
    generated_text = response["choices"][0]["text"].strip()

    # Validate and return the digit
    if generated_text in allowed_digits:
        return generated_text
    else:
        # If the model doesn't return an allowed digit, return None or handle as appropriate
        print(f"Warning: Model generated an unexpected output: '{generated_text}'. Expected one of {allowed_digits}.")
        return None


# ------------------------------------------------------------
# 6. CLASSIFY ONE PROCESS
# ------------------------------------------------------------

def classify_process_text(text):
    # The generative model will process the full document within its n_ctx limit.
    # No need for explicit chunking and aggregation here.
    document = text # Pass the full text directly

    # ========================================================
    # 1. MÍDIAS DIGITAIS
    # ========================================================

    prompt_media = f"""
    Você está analisando um processo judicial brasileiro.

    OBJETIVO:

    Determinar se existe PROVA DIGITAL relacionada aos fatos discutidos no processo.

    Considere apenas evidências digitais utilizadas para demonstrar,
    confirmar, refutar ou contextualizar os fatos controvertidos.

    Considere como prova digital:

    - mensagens de WhatsApp, Telegram, SMS ou similares;
    - e-mails;
    - capturas de tela (prints);
    - fotografias digitais;
    - vídeos;
    - áudios;
    - gravações;
    - publicações em redes sociais;
    - registros de sistemas;
    - logs;
    - metadados;
    - dados extraídos de celulares, computadores ou outros dispositivos;
    - arquivos eletrônicos apresentados como evidência dos fatos;
    - conteúdo armazenado em serviços digitais.

    NÃO considere como prova digital:

    - processo eletrônico;
    - petições eletrônicas;
    - movimentações processuais;
    - documentos assinados digitalmente;
    - certificados digitais;
    - assinaturas eletrônicas;
    - intimações eletrônicas;
    - documentos meramente digitalizados;
    - referências ao sistema do tribunal;
    - e-SAJ, PJe, Projudi ou sistemas equivalentes;
    - atos processuais eletrônicos em geral.

    IMPORTANTE:

    A mera existência de documentos eletrônicos nos autos NÃO significa
    existência de prova digital.

    Classifique como 1 somente quando houver evidência digital relacionada
    aos fatos discutidos no processo.

    Não faça inferências.

    Considere exclusivamente o texto fornecido.

    CLASSIFICAÇÃO:

    0 = Não há evidência de prova digital relacionada aos fatos.

    1 = Há evidência de prova digital relacionada aos fatos.

    Texto do processo:

    {document}

    Responda com apenas o dígito (0 ou 1):
    """

    media_digital = predict_digit(
        prompt_media,
        ["0", "1"]
    )


    # ========================================================
    # 2. IMPUGNAÇÃO DA PROVA DIGITAL
    # ========================================================

    prompt_impugnacao = f"""
    Você está analisando um processo judicial brasileiro.

    OBJETIVO:

    Determinar se alguma das partes apresentou impugnação específica
    contra uma prova digital.

    Considere apenas manifestações expressas que questionem a própria
    confiabilidade da prova digital.

    Exemplos de impugnação específica:

    - questionamento da autenticidade;
    - questionamento da integridade;
    - questionamento da origem;
    - questionamento da autoria;
    - alegação de adulteração;
    - alegação de manipulação;
    - alegação de montagem;
    - alegação de edição;
    - alegação de ausência ou ruptura da cadeia de custódia;
    - questionamento dos métodos de coleta ou extração;
    - questionamento de metadados, logs ou hashes;
    - questionamento da confiabilidade técnica da evidência digital.

    Considere petições, manifestações, recursos, quesitos,
    pareceres técnicos ou outros documentos.

    NÃO considere como impugnação específica:

    - mera discordância sobre os fatos;
    - negativa dos fatos alegados;
    - alegação genérica de insuficiência probatória;
    - alegação de falta de convencimento do juiz;
    - alegação de que a prova não comprova determinada narrativa;
    - discussão jurídica sem questionamento da confiabilidade da prova digital;
    - pedido genérico de produção de provas.

    IMPORTANTE:

    A crítica ao conteúdo da prova não é necessariamente impugnação
    da prova digital.

    Classifique como 1 apenas quando houver questionamento expresso
    da confiabilidade, autenticidade, integridade, origem ou obtenção
    da evidência digital.

    Não faça inferências.

    Considere exclusivamente o texto fornecido.

    CLASSIFICAÇÃO:

    0 = Não há impugnação específica de prova digital.

    1 = Há impugnação específica de prova digital.

    Texto do processo:

    {document}

    Responda com apenas o dígito (0 ou 1):
    """

    impugnacao_digital = predict_digit(
        prompt_impugnacao,
        ["0", "1"]
    )


    # ========================================================
    # 3. CLASSIFICAÇÃO PRINCIPAL
    # ========================================================

    prompt_classificacao = f"""
    Você é um especialista em provas digitais no processo judicial brasileiro.

    Analise exclusivamente as informações presentes no texto.

    Avalie, quando aplicável:

    - cadeia de custódia;
    - origem e identificação da evidência;
    - coleta ou extração;
    - datas e responsáveis;
    - preservação e armazenamento;
    - transferências e acessos;
    - cópias e análises;
    - hashes e integridade;
    - imagens forenses;
    - metadados e logs;
    - autenticidade;
    - método e ferramentas utilizadas;
    - documentação;
    - auditabilidade;
    - contraditório;
    - inconsistências entre documentos.

    REGRAS OBRIGATÓRIAS:

    1. Não faça inferências.

    2. Considere apenas informações expressamente presentes no texto.

    3. A ausência de documentação NÃO significa automaticamente descumprimento.

    4. Porém, a ausência de indícios negativos também NÃO significa aderência.

    5. Classifique como 1 apenas quando existirem elementos positivos
    expressamente descritos nos autos que demonstrem aderência aos parâmetros.

    6. Não utilize presunções de regularidade.

    7. Quando as informações necessárias para avaliação forem insuficientes,
    limitadas ou inconclusivas, classifique como 3.

    8. Diferencie:

    "não consta dos autos"

    de

    "foi demonstrado que não foi realizado".

    9. Para classificar como 2, deve existir evidência objetiva de fragilidade,
    descumprimento, inconsistência relevante ou questionamento fundamentado.

    CLASSIFICAÇÃO:

    1 = Potencialmente Segue.

    Existem evidências positivas e suficientes de aderência aos parâmetros
    relevantes para a prova digital analisada.

    2 = Potencialmente Não Segue.

    Existe evidência objetiva de descumprimento, fragilidade relevante,
    inconsistência ou comprometimento da confiabilidade da prova digital.

    3 = Indecisivo.

    As informações presentes são insuficientes, limitadas ou inconclusivas
    para avaliar aderência ou descumprimento.

    Baseie-se exclusivamente no texto fornecido.

    Texto do processo:

    {document}

    Responda com apenas o dígito (1, 2 ou 3):
    """

    classification = predict_digit(
        prompt_classificacao,
        ["1", "2", "3"]
    )

    print(
        "\n"
        f"MIDIAS_DIGITAIS: {media_digital}\n"
        f"IMPUGNACAO_DA_PROVA_DIGITAL: {impugnacao_digital}\n"
        f"CLASSIFICACAO: {classification}\n"
    )

    return (
        media_digital,
        impugnacao_digital,
        classification
    )

In [ ]:
import os
import pandas as pd

# ------------------------------------------------------------
# 7. FOLDERS / OUTPUT
# ------------------------------------------------------------

root_folder = "/content/drive/MyDrive/ocr_export"

output_csv_path = (
    "/content/drive/MyDrive/"
    "process_classification_results.csv"
)

batch_size = 10


# ------------------------------------------------------------
# 8. CHECK ROOT FOLDER
# ------------------------------------------------------------

if not os.path.exists(root_folder):

    print(
        f"ERROR: Folder not found:\n{root_folder}"
    )

else:

    # --------------------------------------------------------
    # LOAD EXISTING RESULTS
    # --------------------------------------------------------

    existing_results_df = pd.DataFrame()

    if os.path.exists(output_csv_path):

        existing_results_df = pd.read_csv(
            output_csv_path
        )

        print(
            f"Loaded "
            f"{len(existing_results_df)} "
            f"existing results."
        )


    # --------------------------------------------------------
    # PROCESSED IDS
    # --------------------------------------------------------

    if (
        not existing_results_df.empty
        and "Process ID" in existing_results_df.columns
    ):

        processed_ids = set(
            existing_results_df["Process ID"]
            .astype(str)
            .tolist()
        )

    else:

        processed_ids = set()


    # --------------------------------------------------------
    # FIND PROCESS FOLDERS
    # --------------------------------------------------------

    all_process_folders = [
        folder
        for folder in os.listdir(root_folder)
        if os.path.isdir(
            os.path.join(root_folder, folder)
        )
    ]

    unprocessed_folders = [
        folder
        for folder in all_process_folders
        if folder not in processed_ids
    ]


    print(
        f"Found {len(all_process_folders)} "
        f"total process folders."
    )

    print(
        f"{len(unprocessed_folders)} "
        f"are new/unprocessed."
    )


    # ========================================================
    # 9. PROCESS BATCHES
    # ========================================================

    for i in range(
        0,
        len(unprocessed_folders),
        batch_size
    ):

        batch_folders = unprocessed_folders[
            i:i + batch_size
        ]

        batch_results = []

        batch_number = (
            i // batch_size
        ) + 1

        total_batches = (
            len(unprocessed_folders)
            + batch_size
            - 1
        ) // batch_size

        print(
            "\n"
            f"====================================\n"
            f"BATCH {batch_number}/{total_batches}\n"
            f"===================================="
        )


        # ----------------------------------------------------
        # PROCESS EACH FOLDER
        # ----------------------------------------------------

        for process_folder_name in batch_folders:

            process_folder_path = os.path.join(
                root_folder,
                process_folder_name
            )

            print(
                f"\nProcessing: "
                f"{process_folder_name}"
            )


            # ------------------------------------------------
            # READ TXT FILES
            # ------------------------------------------------

            combined_text = []

            for file_name in os.listdir(
                process_folder_path
            ):

                if file_name.lower().endswith(".txt"):

                    file_path = os.path.join(
                        process_folder_path,
                        file_name
                    )

                    try:

                        with open(
                            file_path,
                            "r",
                            encoding="utf-8"
                        ) as f:

                            combined_text.append(
                                f.read()
                            )

                    except Exception as e:

                        print(
                            f"Error reading "
                            f"{file_path}: {e}"
                        )


            full_process_text = "\n".join(
                combined_text
            )


            # ------------------------------------------------
            # CLASSIFY
            # ------------------------------------------------

            if full_process_text.strip():

                try:

                    (
                        media_digital,
                        impugnacao_digital,
                        classification
                    ) = classify_process_text(
                        full_process_text
                    )

                except Exception as e:

                    print(
                        f"ERROR processing "
                        f"{process_folder_name}: {e}"
                    )

                    media_digital = None
                    impugnacao_digital = None
                    classification = None


                batch_results.append({

                    "Process ID":
                        process_folder_name,

                    "MIDIAS_DIGITAIS":
                        media_digital,

                    "IMPUGNACAO_DA_PROVA_DIGITAL":
                        impugnacao_digital,

                    "CLASSIFICACAO":
                        classification
                })


                print(
                    f"RESULT: "
                    f"{media_digital} | "
                    f"{impugnacao_digital} | "
                    f"{classification}"
                )


            # ------------------------------------------------
            # EMPTY PROCESS
            # ------------------------------------------------

            else:

                batch_results.append({

                    "Process ID":
                        process_folder_name,

                    "MIDIAS_DIGITAIS":
                        None,

                    "IMPUGNACAO_DA_PROVA_DIGITAL":
                        None,

                    "CLASSIFICACAO":
                        None
                })

                print(
                    "No text found. "
                    "Values set to NULL."
                )


        # ====================================================
        # 10. SAVE BATCH
        # ====================================================

        if batch_results:

            batch_df = pd.DataFrame(
                batch_results
            )

            file_exists = os.path.exists(
                output_csv_path
            )

            if not file_exists:

                batch_df.to_csv(
                    output_csv_path,
                    index=False,
                    mode="w"
                )

            else:

                batch_df.to_csv(
                    output_csv_path,
                    index=False,
                    mode="a",
                    header=False
                )

            print(
                f"\nBatch saved to:\n"
                f"{output_csv_path}"
            )


    # ========================================================
    # 11. FINAL RESULT
    # ========================================================

    if os.path.exists(
        output_csv_path
    ):

        final_df = pd.read_csv(
            output_csv_path
        )

        print(
            "\n===================================="
        )

        print(
            "PROCESSING COMPLETE"
        )

        print(
            f"Total rows: {len(final_df)}"
        )

        print(
            "Unique processes:",
            final_df["Process ID"].nunique()
        )

        print(
            "\nClassification counts:"
        )

        print(
            final_df["CLASSIFICACAO"]
            .value_counts(dropna=False)
        )

        print(
            "\nCSV:"
        )

        print(
            output_csv_path
        )

    else:

        print(
            "No CSV generated."
        )